# 1D J1J2J3 inference: PoincareRNN (seed 222-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2j3_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # Create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2,J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
syssize = 100
r_max = 0.78
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname = 'results_PoincareRNN'

## J2=0.0, J3=0.5

In [5]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05= -53.99140745384424

In [8]:
seed=111
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.641300201416016-0.0026000000070780516j), var E = 8.518799781799316
DMRG energy  is -53.9914
Time taken=0.155 hrs


In [7]:
seed=222
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 78 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.77539825439453-0.0034000000450760126j), var E = 7.592299938201904
DMRG energy  is -53.9914
Time taken=0.226 hrs


In [6]:
seed=333
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.97840118408203-0.012000000104308128j), var E = 7.265999794006348
DMRG energy  is -53.9914
Time taken=0.433 hrs


In [9]:
seed=444
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 76 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.88890075683594+0.007300000172108412j), var E = 6.922800064086914
DMRG energy  is -53.9914
Time taken=0.212 hrs


In [10]:
seed=555
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 85 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.59519958496094+0.017899999395012856j), var E = 7.447700023651123
DMRG energy  is -53.9914
Time taken=0.208 hrs


## J2=0.2, J3=0.2

In [11]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.58595579948936

In [10]:
seed=111
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.564300537109375-0.006399999838322401j), var E = 3.151900053024292
DMRG energy  is -43.586
Time taken=0.437 hrs


In [10]:
seed=222
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.63779830932617+0.004000000189989805j), var E = 3.455899953842163
DMRG energy  is -43.586
Time taken=0.33 hrs


In [13]:
seed=222
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 111 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.642799377441406+0.004000000189989805j), var E = 3.325500011444092
DMRG energy  is -43.586
Time taken=0.423 hrs


In [14]:
seed=333
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 126 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.710899353027344-0.007000000216066837j), var E = 3.1196000576019287
DMRG energy  is -43.586
Time taken=0.475 hrs


In [7]:
seed=444
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.687400817871094+0.002099999925121665j), var E = 3.2788000106811523
DMRG energy  is -43.586
Time taken=0.529 hrs


In [15]:
seed=444
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 131 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.69160079956055+0.002099999925121665j), var E = 3.134700059890747
DMRG energy  is -43.586
Time taken=0.591 hrs


In [6]:
seed=555
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.661598205566406+0.002300000051036477j), var E = 3.6266000270843506
DMRG energy  is -43.586
Time taken=0.515 hrs


## J2=0.2, J3 = 0.5

In [17]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [12]:
seed=111
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.545799255371094-0.005200000014156103j), var E = 6.439799785614014
DMRG energy  is -49.6287
Time taken=0.448 hrs


In [5]:
seed=222
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.84149932861328-0.006000000052154064j), var E = 6.265500068664551
DMRG energy  is -49.6287
Time taken=0.391 hrs


In [20]:
seed=333
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 103 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.210601806640625-0.008100000210106373j), var E = 7.287799835205078
DMRG energy  is -49.6287
Time taken=0.465 hrs


In [15]:
seed=444
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 78 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-45.38869857788086+0.014399999752640724j), var E = 8.581399917602539
DMRG energy  is -49.6287
Time taken=0.333 hrs


In [8]:
seed=555
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.760501861572266-0.004800000227987766j), var E = 6.4953999519348145
DMRG energy  is -49.6287
Time taken=0.332 hrs


## J2=0.5, J3=0.2

In [21]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [20]:
#POINCARE-RNN: WITH TAU, CLIPPED E VERSION
#Best model saved at epoch 872 with best E=-37.65723+0.05444j, varE=6.81851
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=0.78, seed=111)
h_wl2 = f'{fname_hRNN_tau}/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypRNN_{units}_id_hyp_rmax=0.78_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.85390090942383-0.0006000000284984708j), var E = 0.541700005531311
Successfully remapped and loaded weights.
Clipped 524 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.36479949951172+0.0005000000237487257j), var E = 0.63919997215271
DMRG energy  is -38.5473
Time taken=0.316 hrs


In [14]:
seed=111
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.96269989013672-0.00839999970048666j), var E = 2.3192999362945557
DMRG energy  is -38.5473
Time taken=0.468 hrs


In [22]:
seed=111
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 162 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.962501525878906-0.00839999970048666j), var E = 2.12719988822937
DMRG energy  is -38.5473
Time taken=0.476 hrs


In [23]:
seed=222
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 81 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-27.94230079650879-0.03009999915957451j), var E = 14.678400039672852
DMRG energy  is -38.5473
Time taken=0.471 hrs


In [11]:
seed=333
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.21500015258789-0.0015999999595806003j), var E = 1.1778000593185425
DMRG energy  is -38.5473
Time taken=0.334 hrs


In [24]:
seed=333
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 561 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.23350143432617-0.0015999999595806003j), var E = 0.8159999847412109
DMRG energy  is -38.5473
Time taken=0.485 hrs


In [25]:
seed=444
set_cpu_deterministic(seed)
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 165 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.97359848022461-0.004000000189989805j), var E = 2.2051000595092773
DMRG energy  is -38.5473
Time taken=0.478 hrs


In [13]:
seed=555
units = 80
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max,seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypRNN_80_id_hyp_rmax=0.78_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hRNN, numsamples, h_wl2, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.69430160522461+0.003100000089034438j), var E = 3.41729998588562
DMRG energy  is -38.5473
Time taken=0.328 hrs
